# Notebook 01 — Data Sources & Ethics
## Afghanistan Multi-Index Drought Stress Dashboard

This notebook:
- Defines the core research goal (district-level drought stress monitoring)
- Documents datasets used (NDVI, LST, precipitation, admin boundaries)
- States ethical use and limitations
- Performs a basic connectivity check to Google Earth Engine (GEE)


In [ ]:
# Imports + EE initialization
import ee
import geemap
import pandas as pd

# Initialize Earth Engine
try:
    ee.Initialize()
    print("✅ Earth Engine initialized")
except Exception as e:
    print("❌ Earth Engine init failed. Try re-auth:")
    print("   earthengine authenticate")
    print("   earthengine initialize")
    raise e


✅ Earth Engine initialized


## Selected GEE datasets (v1)

We prioritize **widely used, policy-trusted, long time-series datasets** and work at **district level** for interpretability.

- **NDVI (Vegetation):** MODIS Monthly NDVI — `MODIS/061/MOD13A3`
- **LST (Heat):** MODIS Terra LST 8-day — `MODIS/061/MOD11A2`
- **Rainfall:** CHIRPS Daily — `UCSB-CHG/CHIRPS/DAILY` (we aggregate to monthly ourselves)
- **District boundaries:** FAO GAUL Level 2 — `FAO/GAUL/2015/level2`

Why these choices:
- Monthly/district outputs are stable and easier to communicate for policy
- Daily CHIRPS keeps aggregation transparent and reproducible
- GAUL provides standard administrative units used widely in humanitarian analysis


In [ ]:
# load datasets
ndvi_ic = ee.ImageCollection("MODIS/061/MOD13A3")     # Monthly
lst_ic  = ee.ImageCollection("MODIS/061/MOD11A2")     # 8-day
chirps_ic = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")  # Daily
gaul2_fc = ee.FeatureCollection("FAO/GAUL/2015/level2")

print("✅ Collections created")


✅ Collections created


In [ ]:
# Afghanistan boundary from GAUL level0
gaul0_fc = ee.FeatureCollection("FAO/GAUL/2015/level0")
afg = gaul0_fc.filter(ee.Filter.eq("ADM0_NAME", "Afghanistan")).geometry()

print("✅ Afghanistan geometry loaded")

✅ Afghanistan geometry loaded


In [5]:
# Quick map sanity check
# MODIS NDVI scale factor = 0.0001
ndvi_scaled = sample_ndvi.multiply(0.0001)

ndvi_vis = {
    "min": 0.0,
    "max": 0.8,
    "palette": [
        "#d73027",  # red (low vegetation)
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#66bd63",
        "#1a9850"   # green (healthy vegetation)
    ]
}

Map = geemap.Map(center=[34.5, 66.0], zoom=5)
Map.addLayer(afg, {}, "Afghanistan boundary")
Map.addLayer(ndvi_scaled.clip(afg), ndvi_vis, "NDVI (scaled, June 2020)")
Map


Map(center=[34.5, 66.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## Ethics & limitations

This project uses **environmental indicators** derived from publicly available satellite and climate datasets.

**What this can support**
- District-level monitoring of drought-related stress patterns
- Transparent comparison of stress over time and across districts
- Decision support for prioritization and further assessment

**What this cannot do**
- It cannot infer household-level conditions, individual outcomes, or causality
- It should not be used to identify or target individuals or communities
- Satellite signals are **contextual proxies** and should complement field assessments

**Data responsibility**
- We work at district-scale to reduce misuse risk
- We document dataset assumptions and preprocessing choices
- We report uncertainty and avoid “prediction” framing